# Multilingual reranker benchmark (top-150 dense chunks per query)

Input: dataset `boinhbo/medxir-rerank-candidates` (candidates.parquet) and the packaged models
`boinhbo/medxir-package-rerankers`. No internet. Output: `rerank_benchmark.json`, one `scores_<name>.parquet`
per model (query_id, rank, score).

Set `QUERY_LIMIT` to 100 for a speed and memory check; use 0 for all 1,200 queries.

In [ ]:
import glob
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

QUERY_LIMIT = int(os.environ.get('QUERY_LIMIT', '100'))
MAX_LEN = 1024
BATCH = 16
OUT = Path('/kaggle/working')

models_json = glob.glob('/kaggle/input/**/MODELS.json', recursive=True)
cands_path = glob.glob('/kaggle/input/**/candidates.parquet', recursive=True)
assert len(models_json) == 1 and len(cands_path) == 1, (models_json, cands_path)
MODEL_ROOT = Path(models_json[0]).parent / 'models'
manifest = json.load(open(models_json[0], encoding='utf-8'))['models']

cands = pd.read_parquet(cands_path[0])
if QUERY_LIMIT:
    keep = sorted(cands['query_id'].unique())[:QUERY_LIMIT]
    cands = cands[cands['query_id'].isin(keep)]
print('rows', len(cands), 'queries', cands['query_id'].nunique(), flush=True)
print('gpu', torch.cuda.get_device_name(0), flush=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

QWEN_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the '
               'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
QWEN_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
INSTRUCT = 'Given a Vietnamese medical question, judge whether the document contains the answer to the question'


def score_qwen(model_dir, pairs):
    from transformers import AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, padding_side='left')
    # Qwen3-Reranker is a causal LM: score = P(yes) vs P(no) at the last position (model card).
    model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.bfloat16).cuda().eval()
    yes, no = tok.convert_tokens_to_ids('yes'), tok.convert_tokens_to_ids('no')
    texts = [QWEN_PREFIX + f'<Instruct>: {INSTRUCT}\n<Query>: {q}\n<Document>: {d}' + QWEN_SUFFIX for q, d in pairs]
    out = np.empty(len(texts), dtype=np.float32)
    order = np.argsort([len(t) for t in texts])
    for start in range(0, len(texts), BATCH):
        idx = order[start:start + BATCH]
        enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to('cuda')
        with torch.no_grad():
            logits = model(**enc).logits[:, -1, :]
        two = torch.stack([logits[:, no], logits[:, yes]], dim=1).float()
        out[idx] = torch.log_softmax(two, dim=1)[:, 1].exp().cpu().numpy()
    return out, model


def score_seqcls(model_dir, pairs):
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, torch_dtype=torch.bfloat16).cuda().eval()
    out = np.empty(len(pairs), dtype=np.float32)
    order = np.argsort([len(d) for _, d in pairs])
    for start in range(0, len(pairs), BATCH):
        idx = order[start:start + BATCH]
        enc = tok([pairs[i][0] for i in idx], [pairs[i][1] for i in idx], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out[idx] = model(**enc).logits.view(-1).float().cpu().numpy()
    return out, model


def score_jina(model_dir, pairs):
    # Listwise: all candidates for one query in one context (model card). Returns per-query scores.
    from transformers import AutoModel
    model = AutoModel.from_pretrained(model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True).cuda().eval()
    out = np.empty(len(pairs), dtype=np.float32)
    by_query = {}
    for i, (q, d) in enumerate(pairs):
        by_query.setdefault(q, []).append(i)
    for q, idx in by_query.items():
        docs = [pairs[i][1] for i in idx]
        results = model.rerank(q, docs)
        for r in results:
            out[idx[r['index']]] = r['relevance_score']
    return out, model


SCORERS = {
    'Qwen3-Reranker-0.6B': score_qwen,
    'Qwen3-Reranker-4B': score_qwen,
    'bge-reranker-v2-m3': score_seqcls,
    'jina-reranker-v3.5': score_jina,
}

In [ ]:
results = {}
for name, scorer in SCORERS.items():
    entry = dict(name=name)
    try:
        model_dir = str(MODEL_ROOT / name)
        meta = manifest[[r for r in manifest if r.split('/')[-1] == name][0]]
        entry.update(sha=meta['sha'], created_at=meta['created_at'], license=meta['license'])
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        pairs = list(zip(cands['query'], cands['chunk_text']))
        started = time.perf_counter()
        scores, model = scorer(model_dir, pairs)
        elapsed = time.perf_counter() - started
        entry.update(pairs=len(pairs), seconds=round(elapsed, 1), pairs_per_second=round(len(pairs) / elapsed, 1),
                     peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2), status='ok')
        frame = cands[['query_id', 'rank']].copy()
        frame['score'] = scores
        frame.to_parquet(OUT / f'scores_{name}.parquet', index=False)
        del model
    except Exception as error:  # record and continue with the next model
        entry.update(status='failed', error=f'{type(error).__name__}: {str(error)[:300]}')
    results[name] = entry
    print(json.dumps(entry, ensure_ascii=False), flush=True)
    torch.cuda.empty_cache()

json.dump(dict(query_limit=QUERY_LIMIT, rows=len(cands), results=results),
          open(OUT / 'rerank_benchmark.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('done', flush=True)